<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V8_Phase2_ExperimentFramework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: Experiment-Framework — Anomaly Detection Mannheim
## Cross-City Transfer Learning zur Anomalie-Erkennung (BA)

**Ziel:** Eine konfigurierbare Pipeline die alle Stellschrauben systematisch vergleicht.
Jeder Run ist reproduzierbar, alle Ergebnisse werden in `results_log.csv` gespeichert.

**Experiment-Plan (9 Runs in 4 Sets):**

| Run | Set | Aggregation | Labeling | Modell | Scoring | Hypothese |
|-----|-----|-------------|----------|--------|---------|-----------|
| 1a  | Aggregation | Station-Level | z-Score | LSTM-AE | MSE | Baseline (= V4.1) |
| 1b  | Aggregation | **City-Wide** | z-Score | LSTM-AE | MSE | Haupthebel: mehr Signal |
| 2a  | Labeling | Gewinner 1 | z-Score | LSTM-AE | MSE | Vergleichspunkt |
| 2b  | Labeling | Gewinner 1 | **Poisson-ppf** | LSTM-AE | MSE | Korrekte Verteilungsannahme |
| 2c  | Labeling | Gewinner 1 | **IQR×3** | LSTM-AE | MSE | Robust, verteilungsfrei |
| 3a  | Modell | Gewinner 1+2 | Gewinner 2 | LSTM-AE | MSE | Vergleichspunkt |
| 3b  | Modell | Gewinner 1+2 | Gewinner 2 | **IsolationForest** | Score | Shallow vs. Deep |
| 4a  | Scoring | Bestes Modell | Bestes Label | Best | **MSE** | Vergleichspunkt |
| 4b  | Scoring | Bestes Modell | Bestes Label | Best | **MAE** | Robuster bei Ausreißern |

In [1]:
# ══════════════════════════════════════════════════════════════
# 0 — Google Drive & Setup
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/BA_Colab/data.zip" "/content/data.zip"
!unzip -q "/content/data.zip" -d "/content"
!rm "/content/data.zip"
!rm -rf "/content/_MACOSX"

Mounted at /content/drive


In [22]:
# ══════════════════════════════════════════════════════════════
# 0b — Cleaning: Erstellt /content/data/cleaned/
#
# Maßnahmen (basierend auf Audit-Ergebnissen):
#   [1] geo_information: WKB-Hex → lat/lon dekodieren, 'location' droppen
#   [2] demand/Mannheim: nur station_type='real' behalten
#   [3] demand/Mannheim: Stationen mit max. Datenlücke > 90 Tage entfernen
#   [4] demand/Mannheim: fehlende Tage (Demand-Lücken auf Stadtebene) entfernen
# ══════════════════════════════════════════════════════════════

import os, glob, re
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from shapely import wkb as shapely_wkb

DATA_BASE    = '/content/data'
CLEANED_BASE = '/content/data/cleaned'
CITY         = 'Mannheim'

def classify_station(name):
    if not isinstance(name, str) or name.strip() == '': return 'unknown'
    n = name.strip()
    if re.search(r'(?i)^recording[_\-\s]', n): return 'recording'
    if re.match(r'(?i)^bike[-_]?\s*\d*', n):   return 'bike'
    if re.search(r'(?i)(virtuell|virtual)', n): return 'virtual'
    if re.fullmatch(r'[\d\s\-_\.#/]+', n):     return 'only_nums'
    return 'real'

station_names_raw = pd.read_parquet(f'{DATA_BASE}/station_names/station_names.parquet')
station_names_raw = station_names_raw.rename(columns={'id': 'station_name_id', 'name': 'station_name'})
station_names_raw['station_type'] = station_names_raw['station_name'].apply(classify_station)
type_lookup = station_names_raw.set_index('station_name_id')['station_type'].to_dict()

print('╔══════════════════════════════════════════════════╗')
print('║  CLEANING — Erstelle /content/data/cleaned/      ║')
print('╚══════════════════════════════════════════════════╝')

# [1] GEO_INFORMATION: WKB → lat/lon
print('\n[1] geo_information: WKB → lat/lon ...')
geo_out_dir = f'{CLEANED_BASE}/geo_information'
os.makedirs(geo_out_dir, exist_ok=True)
geo_raw = pq.read_table(f'{DATA_BASE}/geo_information').to_pandas()
geom = geo_raw['location'].apply(lambda x: shapely_wkb.loads(bytes.fromhex(x)))
geo_raw['latitude']  = geom.apply(lambda g: round(g.y, 6))
geo_raw['longitude'] = geom.apply(lambda g: round(g.x, 6))
geo_clean = geo_raw.drop(columns=['location'])
geo_clean.to_parquet(f'{geo_out_dir}/geo_information.parquet', index=False)
print(f'  ✅ {len(geo_clean):,} Einträge, Spalten: {list(geo_clean.columns)}')

# [2+3+4] DEMAND / MANNHEIM
print(f'\n[2-4] demand/{CITY}: Laden ...')
files = glob.glob(f'{DATA_BASE}/demand/{CITY}/**/*.parquet', recursive=True)
if not files:
    files = glob.glob(f'{DATA_BASE}/demand/{CITY}/*.parquet')

cols = ['network_name', 'timestamp', 'station_id', 'station_name_id',
        'location_id', 'n_lends', 'n_returns']
demand = pd.concat([pd.read_parquet(f, columns=cols) for f in files], ignore_index=True)
demand['timestamp']    = pd.to_datetime(demand['timestamp'], utc=True)
demand['station_type'] = demand['station_name_id'].map(type_lookup).fillna('unknown')
demand['total_demand'] = demand['n_lends'] + demand['n_returns']

# [2] Nur real
demand = demand[demand['station_type'] == 'real'].copy()

# [3] Lücken > 90 Tage raus
demand['date'] = demand['timestamp'].dt.date
def max_gap(dates):
    s = sorted(set(pd.to_datetime(d) for d in dates))
    if len(s) < 2: return 0
    return max((s[i+1] - s[i]).days - 1 for i in range(len(s)-1))
station_gaps = demand.groupby('station_id')['date'].apply(max_gap).reset_index()
station_gaps.columns = ['station_id', 'max_gap_days']
bad_stations = station_gaps[station_gaps['max_gap_days'] > 90]['station_id'].tolist()
demand = demand[~demand['station_id'].isin(bad_stations)].copy()

# [4] Fehlende Tage raus
daily_city = demand.groupby('date')['total_demand'].sum().reset_index()
all_dates  = pd.date_range(demand['timestamp'].min().date(),
                           demand['timestamp'].max().date(), freq='D')
existing   = set(pd.Timestamp(d) for d in daily_city['date'])
missing    = {d.date() for d in all_dates if d not in existing}
if missing:
    demand = demand[~demand['date'].isin(missing)].copy()

demand = demand.drop(columns=['station_type', 'total_demand', 'date'])
demand_out_dir = f'{CLEANED_BASE}/demand/{CITY}'
os.makedirs(demand_out_dir, exist_ok=True)
demand.to_parquet(f'{demand_out_dir}/demand_cleaned.parquet', index=False)

print(f'  ✅ {len(demand):,} Zeilen | {demand["station_id"].nunique()} Stationen')
print(f'  Entfernt: {len(bad_stations)} Stationen (Lücke>90d), {len(missing)} fehlende Tage')
print('\n✅ Cleaning abgeschlossen.')

╔══════════════════════════════════════════════════╗
║  CLEANING — Erstelle /content/data/cleaned/      ║
╚══════════════════════════════════════════════════╝

[1] geo_information: WKB → lat/lon ...
  ✅ 300,508 Einträge, Spalten: ['location_id', 'continent_name', 'country_name', 'city_name', 'federal_state_name', 'postal_code', 'elevation', 'latitude', 'longitude']

[2-4] demand/Mannheim: Laden ...


KeyboardInterrupt: 

In [23]:
# ══════════════════════════════════════════════════════════════
# 1 — Imports & globale Konstanten
# ══════════════════════════════════════════════════════════════
import os, warnings, json
from dataclasses import dataclass, field, asdict
from typing import Literal, List
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_recall_curve, auc, f1_score,
    roc_auc_score, average_precision_score
)
from scipy.stats import poisson
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  {torch.cuda.get_device_name(0)}')

# ── Pfade ──
CLEANED_BASE       = '/content/data/cleaned'
DATA_BASE          = '/content/data'
DEMAND_PATH        = f'{CLEANED_BASE}/demand/Mannheim/demand_cleaned.parquet'
STATION_NAMES_PATH = f'{DATA_BASE}/station_names/station_names.parquet'
GEO_INFO_PATH      = f'{CLEANED_BASE}/geo_information/geo_information.parquet'
WEATHER_PATH       = f'{DATA_BASE}/weather/weather.parquet'
HOLIDAYS_PATH      = f'{DATA_BASE}/holidays/holidays.parquet'
VACATIONS_PATH     = f'{DATA_BASE}/vacations/vacations.parquet'

RESULTS_LOG        = '/content/data/results_log.csv'
PLOTS_DIR          = '/content/data/experiment_plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

# ── Globale Konstanten (nicht per Experiment variiert) ──
CITY             = 'Mannheim'
FEDERAL_STATE    = 'Baden-Württemberg'
WEATHER_LOC_ID   = 292348
TRAIN_RATIO      = 0.67
VAL_RATIO        = 0.83
EPOCHS           = 50
BATCH_SIZE       = 2048
LEARNING_RATE    = 1e-3
LATENT_DIM       = 32
EARLY_STOP       = 8

print('Setup abgeschlossen.')

Device: cuda
  NVIDIA L4
Setup abgeschlossen.


In [24]:
# ══════════════════════════════════════════════════════════════
# 2 — ExperimentConfig Dataclass
#
# Alle Stellschrauben an einem Ort. Jeder Run ist vollständig
# durch ein Config-Objekt beschrieben → reproduzierbar.
# ══════════════════════════════════════════════════════════════

@dataclass
class ExperimentConfig:
    # ── Identifikation ──
    run_id:       str   = 'run_1a'
    description:  str   = 'Baseline (Station-Level, z-Score, LSTM-AE, MSE)'
    experiment_set: str = 'Set1_Aggregation'

    # ── Aggregation ──
    # 'station':    Pro Station hourly (V4.1 Baseline)
    # 'city_wide':  Summe aller Stationen pro Stunde
    aggregation:  Literal['station', 'city_wide'] = 'station'

    # ── Labeling ──
    # 'zscore':     |z| > z_threshold AND hist_mean >= min_hist_mean AND demand >= min_absolute
    # 'poisson':    demand > poisson.ppf(poisson_quantile, hist_mean) AND hist_mean >= min_hist_mean
    # 'iqr':        demand > Q3 + iqr_factor * IQR (per station×dow×hour)
    labeling:     Literal['zscore', 'poisson', 'iqr'] = 'zscore'
    z_threshold:      float = 3.0
    z_train_threshold: float = 2.0   # Grauzone-Untergrenze
    poisson_quantile: float = 0.9987  # entspricht z=3 für Gauß
    iqr_factor:       float = 3.0
    min_hist_mean:    float = 2.0
    min_absolute:     int   = 5

    # ── Features ──
    # 'base':    Demand-Roh + Lags + Kalender + Wetter (19 Features, V4 Standard)
    # 'minimal': Nur Demand + Kalender (ohne Wetter, für Transfer robuster)
    feature_set:  Literal['base', 'minimal'] = 'base'

    # ── Modell ──
    # 'lstm_ae':        LSTM-Autoencoder (sequentiell)
    # 'vanilla_ae':     Flacher MLP-Autoencoder (Baseline)
    # 'isolation_forest': Sklearn, kein Deep Learning
    model:        Literal['lstm_ae', 'vanilla_ae', 'isolation_forest'] = 'lstm_ae'
    window_size:  int   = 24   # Stunden (nur für AE-Modelle)
    if_contamination: float = 0.005  # für IsolationForest: geschätzte Anomalie-Rate
    if_n_estimators:  int   = 200

    # ── Scoring (nur für AE-Modelle) ──
    # 'mse': Mean Squared Error über Demand-Features
    # 'mae': Mean Absolute Error über Demand-Features
    scoring:      Literal['mse', 'mae'] = 'mse'

    # ── Stationsfilter ──
    min_events_per_day: float = 3.0  # Stationen unter dieser Schwelle raus


# ── Die 9 Experiment-Configs ──
EXPERIMENTS = [
    # Set 1: Aggregation
    ExperimentConfig(
        run_id='1a', description='Baseline: Station-Level, z-Score, LSTM-AE, MSE',
        experiment_set='Set1_Aggregation',
        aggregation='station', labeling='zscore', model='lstm_ae', scoring='mse'
    ),
    ExperimentConfig(
        run_id='1b', description='City-Wide Aggregation, z-Score, LSTM-AE, MSE',
        experiment_set='Set1_Aggregation',
        aggregation='city_wide', labeling='zscore', model='lstm_ae', scoring='mse'
    ),
    # Set 2: Labeling — Aggregation-Gewinner wird unten eingesetzt
    ExperimentConfig(
        run_id='2a', description='Labeling-Vergleich: z-Score (Referenz)',
        experiment_set='Set2_Labeling',
        aggregation='WINNER_SET1',  # wird vor dem Run durch Gewinner ersetzt
        labeling='zscore', model='lstm_ae', scoring='mse'
    ),
    ExperimentConfig(
        run_id='2b', description='Labeling: Poisson-ppf(0.9987)',
        experiment_set='Set2_Labeling',
        aggregation='WINNER_SET1',
        labeling='poisson', model='lstm_ae', scoring='mse'
    ),
    ExperimentConfig(
        run_id='2c', description='Labeling: IQR x 3.0',
        experiment_set='Set2_Labeling',
        aggregation='WINNER_SET1',
        labeling='iqr', iqr_factor=3.0, model='lstm_ae', scoring='mse'
    ),
    # Set 3: Modell
    ExperimentConfig(
        run_id='3a', description='Modell-Vergleich: LSTM-AE (Referenz)',
        experiment_set='Set3_Model',
        aggregation='WINNER_SET1', labeling='WINNER_SET2',
        model='lstm_ae', scoring='mse'
    ),
    ExperimentConfig(
        run_id='3b', description='Modell: Isolation Forest',
        experiment_set='Set3_Model',
        aggregation='WINNER_SET1', labeling='WINNER_SET2',
        model='isolation_forest', scoring='mse'
    ),
    # Set 4: Scoring
    ExperimentConfig(
        run_id='4a', description='Scoring-Vergleich: MSE (Referenz)',
        experiment_set='Set4_Scoring',
        aggregation='WINNER_SET1', labeling='WINNER_SET2',
        model='WINNER_SET3', scoring='mse'
    ),
    ExperimentConfig(
        run_id='4b', description='Scoring: MAE (robust)',
        experiment_set='Set4_Scoring',
        aggregation='WINNER_SET1', labeling='WINNER_SET2',
        model='WINNER_SET3', scoring='mae'
    ),
]

print(f'{len(EXPERIMENTS)} Experiment-Configs definiert.')
for e in EXPERIMENTS:
    print(f'  [{e.run_id}] {e.description}')

9 Experiment-Configs definiert.
  [1a] Baseline: Station-Level, z-Score, LSTM-AE, MSE
  [1b] City-Wide Aggregation, z-Score, LSTM-AE, MSE
  [2a] Labeling-Vergleich: z-Score (Referenz)
  [2b] Labeling: Poisson-ppf(0.9987)
  [2c] Labeling: IQR x 3.0
  [3a] Modell-Vergleich: LSTM-AE (Referenz)
  [3b] Modell: Isolation Forest
  [4a] Scoring-Vergleich: MSE (Referenz)
  [4b] Scoring: MAE (robust)


In [25]:
# ══════════════════════════════════════════════════════════════
# 3 — Hilfsdaten laden (einmalig, für alle Experimente)
# ══════════════════════════════════════════════════════════════

# ── Wetter ──
weather = pd.read_parquet(WEATHER_PATH)
weather['timestamp'] = pd.to_datetime(weather['timestamp'], utc=True)
weather_ma = weather[weather['location_id'] == WEATHER_LOC_ID].copy()
weather_ma['hour_ts'] = weather_ma['timestamp'].dt.floor('h')
weather_hourly = (
    weather_ma.groupby('hour_ts')
    .agg(temperature=('temperature','mean'), humidity=('humidity','mean'),
         precipitation=('precipitation','sum'), wind_speed=('wind_speed','mean'))
    .reset_index()
)

# ── Feiertage & Ferien (BaWü) ──
holidays  = pd.read_parquet(HOLIDAYS_PATH)
vacations = pd.read_parquet(VACATIONS_PATH)
for df in [holidays, vacations]:
    df['start_date'] = pd.to_datetime(df['start_date'])
    df['end_date']   = pd.to_datetime(df['end_date'])

holiday_dates = set()
for _, row in holidays[holidays['federal_state_name'] == FEDERAL_STATE].iterrows():
    for d in pd.date_range(row['start_date'], row['end_date']):
        holiday_dates.add(d.date())

vacation_dates = set()
for _, row in vacations[vacations['federal_state_name'] == FEDERAL_STATE].iterrows():
    for d in pd.date_range(row['start_date'], row['end_date']):
        vacation_dates.add(d.date())

print(f'Wetter: {len(weather_hourly):,} Stunden | '
      f'{weather_hourly["hour_ts"].min().date()} – {weather_hourly["hour_ts"].max().date()}')
print(f'Feiertage: {len(holiday_dates)} | Ferien: {len(vacation_dates)}')

Wetter: 24,913 Stunden | 2023-04-01 – 2026-02-02
Feiertage: 167 | Ferien: 277


In [26]:
# ══════════════════════════════════════════════════════════════
# 4 — Daten-Loader: Station-Level & City-Wide
# ══════════════════════════════════════════════════════════════

def load_demand_base() -> pd.DataFrame:
    """Lädt gecleante Demand-Daten, aggregiert auf Stundenbasis pro Station."""
    demand = pd.read_parquet(DEMAND_PATH)
    demand['timestamp']    = pd.to_datetime(demand['timestamp'], utc=True)
    demand['total_demand'] = demand['n_lends'] + demand['n_returns']
    demand['hour_ts']      = demand['timestamp'].dt.floor('h')

    # Stündliche Aggregation pro Station
    hourly = (
        demand.groupby(['station_id', 'station_name_id', 'location_id', 'hour_ts'])
        .agg(n_lends=('n_lends','sum'), n_returns=('n_returns','sum'))
        .reset_index()
    )
    hourly['total_demand'] = hourly['n_lends'] + hourly['n_returns']

    # Deduplizierung (station_id × hour_ts)
    hourly = (
        hourly.groupby(['station_id','hour_ts'])
        .agg(n_lends=('n_lends','sum'), n_returns=('n_returns','sum'),
             total_demand=('total_demand','sum'),
             station_name_id=('station_name_id','first'),
             location_id=('location_id','first'))
        .reset_index()
    )

    # Lückenfüllung
    all_hours = pd.date_range(hourly['hour_ts'].min(), hourly['hour_ts'].max(),
                              freq='h', tz='UTC')
    station_info = (
        hourly.groupby('station_id')
        .agg(station_name_id=('station_name_id','first'),
             location_id=('location_id','first'))
        .reset_index()
    )
    full_idx = pd.MultiIndex.from_product(
        [station_info['station_id'].values, all_hours],
        names=['station_id','hour_ts']
    )
    hourly_full = (
        hourly[['station_id','hour_ts','n_lends','n_returns','total_demand']]
        .set_index(['station_id','hour_ts'])
        .reindex(full_idx, fill_value=0)
        .reset_index()
    )
    hourly_full = hourly_full.merge(station_info, on='station_id', how='left')
    return hourly_full


def build_station_level(hourly_full: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    """Kalender + Wetter Features für station-level Daten."""
    df = hourly_full.copy()

    # Stationsfilter
    n_days = (df['hour_ts'].max() - df['hour_ts'].min()).days + 1
    min_events = int(n_days * cfg.min_events_per_day)
    active = df.groupby('station_id')['total_demand'].sum()
    df = df[df['station_id'].isin(active[active >= min_events].index)].copy()

    df = _add_calendar_features(df)
    df = _add_weather_features(df)
    df = _add_lag_features(df, group_col='station_id')
    return df


def build_city_wide(hourly_full: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    """Aggregiert alle Stationen zur Stadtebene → eine Zeitreihe."""
    city = (
        hourly_full.groupby('hour_ts')
        .agg(n_lends=('n_lends','sum'), n_returns=('n_returns','sum'),
             total_demand=('total_demand','sum'))
        .reset_index()
    )
    city['station_id'] = 'city_wide'  # Pseudo-ID für einheitliche Pipeline

    city = _add_calendar_features(city)
    city = _add_weather_features(city)
    city = _add_lag_features(city, group_col='station_id')
    return city


def _add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    df['hour_of_day'] = df['hour_ts'].dt.hour
    df['day_of_week'] = df['hour_ts'].dt.dayofweek
    df['month']       = df['hour_ts'].dt.month
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(np.int8)
    df['is_holiday']  = df['hour_ts'].dt.date.apply(lambda x: 1 if x in holiday_dates else 0).astype(np.int8)
    df['is_vacation'] = df['hour_ts'].dt.date.apply(lambda x: 1 if x in vacation_dates else 0).astype(np.int8)
    df['hour_sin']    = np.sin(2 * np.pi * df['hour_of_day'] / 24)
    df['hour_cos']    = np.cos(2 * np.pi * df['hour_of_day'] / 24)
    df['dow_sin']     = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']     = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)
    return df


def _add_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.merge(weather_hourly, on='hour_ts', how='left')
    for col in ['temperature', 'humidity', 'precipitation', 'wind_speed']:
        df[col] = df[col].interpolate(method='linear', limit=6).ffill().bfill()
        df[col] = df[col].fillna(df[col].median())
    return df


def _add_lag_features(df: pd.DataFrame, group_col: str = 'station_id') -> pd.DataFrame:
    df = df.sort_values([group_col, 'hour_ts']).reset_index(drop=True)
    for lag_name, lag_h in [('lag_1h', 1), ('lag_24h', 24), ('lag_168h', 168)]:
        df[f'demand_{lag_name}'] = df.groupby(group_col)['total_demand'].shift(lag_h)
    df = df.dropna(subset=['demand_lag_168h']).reset_index(drop=True)
    return df


def get_feature_cols(cfg: ExperimentConfig) -> list:
    base = [
        'total_demand', 'n_lends', 'n_returns',
        'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
        'is_weekend', 'is_holiday', 'is_vacation',
        'temperature', 'humidity', 'precipitation', 'wind_speed',
    ]
    minimal = [
        'total_demand', 'n_lends', 'n_returns',
        'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
        'is_weekend', 'is_holiday', 'is_vacation',
    ]
    return base if cfg.feature_set == 'base' else minimal


print('Daten-Loader Funktionen definiert.')

Daten-Loader Funktionen definiert.


In [27]:
# ══════════════════════════════════════════════════════════════
# 5 — Labeling-Strategien
#
# WICHTIG: Stats werden IMMER nur aus Trainingsperiode berechnet.
# Alle drei Methoden erzeugen: label ∈ {'normal', 'grauzone', 'anomal'}
# ══════════════════════════════════════════════════════════════

def compute_labels(df: pd.DataFrame, train_end: pd.Timestamp,
                   cfg: ExperimentConfig, group_col: str = 'station_id') -> pd.DataFrame:
    """
    Berechnet Labels (normal / grauzone / anomal) auf Basis der Config.
    Stats werden ausschließlich aus df[hour_ts < train_end] berechnet.
    """
    train = df[df['hour_ts'] < train_end]
    df = df.copy()
    df['label'] = 'normal'

    if cfg.labeling == 'zscore':
        stats = (
            train.groupby([group_col, 'day_of_week', 'hour_of_day'])['total_demand']
            .agg(['mean', 'std'])
            .rename(columns={'mean': 'hist_mean', 'std': 'hist_std'})
            .reset_index()
        )
        df = df.merge(stats, on=[group_col, 'day_of_week', 'hour_of_day'], how='left')
        gm, gs = train['total_demand'].mean(), train['total_demand'].std()
        df['hist_mean'] = df['hist_mean'].fillna(gm)
        df['hist_std']  = df['hist_std'].fillna(gs).clip(lower=0.1)
        df['_score']    = (df['total_demand'] - df['hist_mean']) / df['hist_std']

        is_anomaly = (
            (df['_score'].abs() > cfg.z_threshold) &
            (df['hist_mean'] >= cfg.min_hist_mean) &
            (df['total_demand'] >= cfg.min_absolute)
        )
        is_grauzone = (df['_score'].abs() > cfg.z_train_threshold) & ~is_anomaly

    elif cfg.labeling == 'poisson':
        # Poisson-ppf: korrekte Verteilungsannahme für Count-Daten
        # Referenz: Ihler et al. (2006) — Adaptive Event Detection with Time-Varying Poisson Processes
        stats = (
            train.groupby([group_col, 'day_of_week', 'hour_of_day'])['total_demand']
            .mean()
            .rename('hist_mean')
            .reset_index()
        )
        df = df.merge(stats, on=[group_col, 'day_of_week', 'hour_of_day'], how='left')
        df['hist_mean'] = df['hist_mean'].fillna(train['total_demand'].mean()).clip(lower=0.01)

        # Obere und untere Poisson-Grenze
        df['_poisson_upper'] = df['hist_mean'].apply(
            lambda lam: poisson.ppf(cfg.poisson_quantile, lam)
        )
        df['_poisson_lower'] = df['hist_mean'].apply(
            lambda lam: poisson.ppf(1 - cfg.poisson_quantile, lam)
        )
        # Grauzone: zwischen ppf(0.95) und ppf(0.9987)
        _gz_upper = df['hist_mean'].apply(lambda lam: poisson.ppf(0.95, lam))

        is_anomaly = (
            (df['total_demand'] > df['_poisson_upper']) &
            (df['hist_mean'] >= cfg.min_hist_mean) &
            (df['total_demand'] >= cfg.min_absolute)
        )
        is_grauzone = (
            (df['total_demand'] > _gz_upper) &
            (df['total_demand'] <= df['_poisson_upper'])
        )

    elif cfg.labeling == 'iqr':
        # IQR: robust, keine Verteilungsannahme
        stats = (
            train.groupby([group_col, 'day_of_week', 'hour_of_day'])['total_demand']
            .agg(q1=lambda x: x.quantile(0.25),
                 q3=lambda x: x.quantile(0.75))
            .reset_index()
        )
        stats['iqr']         = stats['q3'] - stats['q1']
        stats['upper_fence'] = stats['q3'] + cfg.iqr_factor * stats['iqr']
        stats['lower_fence'] = stats['q1'] - cfg.iqr_factor * stats['iqr']
        # Grauzone: 1.5×IQR Fence (Tukey mild) bis 3.0×IQR Fence (Tukey extreme)
        stats['mild_upper']  = stats['q3'] + 1.5 * stats['iqr']

        df = df.merge(stats, on=[group_col, 'day_of_week', 'hour_of_day'], how='left')
        # Fallback für fehlende Stats (neue Stationen / Slots)
        global_q1 = train['total_demand'].quantile(0.25)
        global_q3 = train['total_demand'].quantile(0.75)
        global_iqr = global_q3 - global_q1
        df['upper_fence'] = df['upper_fence'].fillna(global_q3 + cfg.iqr_factor * global_iqr)
        df['mild_upper']  = df['mild_upper'].fillna(global_q3 + 1.5 * global_iqr)

        is_anomaly = (
            (df['total_demand'] > df['upper_fence']) &
            (df['total_demand'] >= cfg.min_absolute)
        )
        is_grauzone = (
            (df['total_demand'] > df['mild_upper']) &
            (df['total_demand'] <= df['upper_fence'])
        )

    df.loc[is_grauzone, 'label'] = 'grauzone'
    df.loc[is_anomaly,  'label'] = 'anomal'

    # Cleanup temp-Spalten
    drop_cols = [c for c in df.columns if c.startswith('_') or c in
                 ('q1','q3','iqr','upper_fence','lower_fence','mild_upper',
                  '_poisson_upper','_poisson_lower')]
    df = df.drop(columns=drop_cols, errors='ignore')
    return df


print('Labeling-Funktionen definiert (zscore / poisson / iqr).')

Labeling-Funktionen definiert (zscore / poisson / iqr).


In [28]:
# ══════════════════════════════════════════════════════════════
# 6 — Modelle
# ══════════════════════════════════════════════════════════════

class LSTMAutoencoder(nn.Module):
    """Sequence-to-Sequence LSTM Autoencoder."""
    def __init__(self, n_features: int, latent_dim: int = 32, n_layers: int = 2):
        super().__init__()
        self.n_features = n_features
        self.encoder = nn.LSTM(n_features, latent_dim, n_layers,
                               batch_first=True, dropout=0.1 if n_layers > 1 else 0)
        self.decoder = nn.LSTM(latent_dim, latent_dim, n_layers,
                               batch_first=True, dropout=0.1 if n_layers > 1 else 0)
        self.output_layer = nn.Linear(latent_dim, n_features)

    def forward(self, x):
        # x: (batch, seq_len, features)
        _, (h, c) = self.encoder(x)
        # Repeated context vector als Decoder-Input
        repeated = h[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        out, _ = self.decoder(repeated, (h, c))
        return self.output_layer(out)


class VanillaAE(nn.Module):
    """Flacher MLP-Autoencoder (Baseline)."""
    def __init__(self, input_dim: int, latent_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),       nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 128),        nn.ReLU(),
            nn.Linear(128, input_dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))


print('Modell-Klassen definiert (LSTMAutoencoder, VanillaAE).')

Modell-Klassen definiert (LSTMAutoencoder, VanillaAE).


In [29]:
# ══════════════════════════════════════════════════════════════
# 7 — Training & Scoring Funktionen
# ══════════════════════════════════════════════════════════════

def train_ae(model, X_train, X_val, model_name='AE',
             epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
             patience=EARLY_STOP, val_chunk=2048):
    """val_chunk: Val-Loss chunk-weise — kein OOM bei großen Val-Sets."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    criterion = nn.MSELoss()

    X_t = torch.FloatTensor(X_train)
    loader = DataLoader(TensorDataset(X_t), batch_size=batch_size, shuffle=True)
    X_v_cpu = torch.FloatTensor(X_val)  # bleibt auf CPU

    best_val, best_state, patience_counter = float('inf'), None, 0

    for epoch in range(epochs):
        model.train()
        losses = []
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out  = model(batch)
            loss = criterion(out, batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        # Val-Loss chunk-weise — nie >val_chunk Sequenzen auf GPU gleichzeitig
        model.eval()
        val_losses_chunks = []
        with torch.no_grad():
            for i in range(0, len(X_v_cpu), val_chunk):
                chunk = X_v_cpu[i:i+val_chunk].to(device)
                val_losses_chunks.append(criterion(model(chunk), chunk).item())
                del chunk
        val_loss = float(np.mean(val_losses_chunks))
        torch.cuda.empty_cache()

        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0:
            print(f'  [{model_name}] Epoch {epoch+1:3d} | '
                  f'Train: {np.mean(losses):.5f} | Val: {val_loss:.5f}')
        if patience_counter >= patience:
            print(f'  [{model_name}] Early stop @ epoch {epoch+1}')
            break

    if best_state:
        model.load_state_dict(best_state)
    model = model.to(device)
    print(f'  [{model_name}] Best Val Loss: {best_val:.6f}')
    return model


def compute_ae_scores(model, X, scoring='mse',
                      demand_indices=None, window_size=24,
                      n_features=None, chunk_size=2048):
    """
    Reconstruction Error (MSE oder MAE) auf Demand-Features.
    Unterstützt sowohl LSTM (3D) als auch Vanilla AE (2D).
    """
    model.eval()
    model = model.to(device)
    all_scores = []

    for i in range(0, len(X), chunk_size):
        chunk = torch.FloatTensor(X[i:i+chunk_size]).to(device)
        with torch.no_grad():
            x_hat = model(chunk)

        if chunk.dim() == 3:
            c_d = chunk[:, :, demand_indices]
            h_d = x_hat[:, :, demand_indices]
        else:
            # Flat → reshape → slice
            c_3d = chunk.reshape(-1, window_size, n_features)
            h_3d = x_hat.reshape(-1, window_size, n_features)
            c_d  = c_3d[:, :, demand_indices]
            h_d  = h_3d[:, :, demand_indices]

        if scoring == 'mse':
            err = (c_d - h_d) ** 2
        else:  # mae
            err = (c_d - h_d).abs()

        scores = err.mean(dim=tuple(range(1, err.dim())))
        all_scores.append(scores.cpu())
        torch.cuda.empty_cache()

    return torch.cat(all_scores).numpy()


def find_best_threshold(scores, labels):
    binary = (np.array(labels) == 'anomal').astype(int)
    if binary.sum() == 0:
        return np.percentile(scores, 95), 0.0
    prec, rec, thresholds = precision_recall_curve(binary, scores)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_idx = np.argmax(f1[:-1])
    return thresholds[best_idx], f1[best_idx]


print('Training- und Scoring-Funktionen definiert.')

Training- und Scoring-Funktionen definiert.


In [30]:
# ══════════════════════════════════════════════════════════════
# 8 — run_experiment(config): Hauptfunktion
#
# Nimmt eine ExperimentConfig, führt den vollständigen Run durch
# und gibt ein Results-Dict zurück. Speichert automatisch in
# results_log.csv.
# ══════════════════════════════════════════════════════════════

def run_experiment(cfg: ExperimentConfig, hourly_raw: pd.DataFrame) -> dict:
    print(f'\n{"="*65}')
    print(f'RUN [{cfg.run_id}] — {cfg.description}')
    print(f'{"="*65}')

    # ── 1. Daten vorbereiten ──
    if cfg.aggregation == 'city_wide':
        df = build_city_wide(hourly_raw, cfg)
        group_col = 'station_id'  # ist 'city_wide'
    else:
        df = build_station_level(hourly_raw, cfg)
        group_col = 'station_id'

    feature_cols = get_feature_cols(cfg)
    # Für city_wide: n_lends, n_returns sind vorhanden → kein Problem
    # Sicherstellen dass alle feature_cols existieren
    feature_cols = [c for c in feature_cols if c in df.columns]
    demand_indices = [i for i, c in enumerate(feature_cols)
                      if c in ('total_demand', 'n_lends', 'n_returns',
                               'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h')]

    n_features = len(feature_cols)
    print(f'Features: {n_features} | Aggregation: {cfg.aggregation}')
    print(f'Stationen/Serien: {df[group_col].nunique()} | Zeilen: {len(df):,}')

    # ── 2. Temporaler Split ──
    t_min  = df['hour_ts'].min()
    t_max  = df['hour_ts'].max()
    total_h = (t_max - t_min).total_seconds() / 3600
    train_end = t_min + pd.Timedelta(hours=int(total_h * TRAIN_RATIO))
    val_end   = t_min + pd.Timedelta(hours=int(total_h * VAL_RATIO))
    print(f'Split: Train bis {train_end.date()} | Val bis {val_end.date()} | Test bis {t_max.date()}')

    # ── 3. Labeling ──
    df = compute_labels(df, train_end, cfg, group_col=group_col)
    label_dist = df['label'].value_counts()
    anomaly_rate = (df['label'] == 'anomal').mean()
    print(f'Labels: {label_dist.to_dict()} | Anomalie-Rate: {anomaly_rate:.4f}')

    if (df['label'] == 'anomal').sum() < 10:
        print('⚠️  Zu wenige Anomalien für sinnvolle Evaluation (< 10). Run wird übersprungen.')
        return {'run_id': cfg.run_id, 'error': 'too_few_anomalies'}

    # ── 4. Train / Val / Test Split ──
    df_train = df[(df['hour_ts'] < train_end) & (df['label'] == 'normal')].copy()
    df_val   = df[(df['hour_ts'] >= train_end) & (df['hour_ts'] < val_end)
                  & (df['label'] != 'grauzone')].copy()
    df_test  = df[(df['hour_ts'] >= val_end) & (df['label'] != 'grauzone')].copy()

    print(f'Train (normal): {len(df_train):,} | '
          f'Val: {len(df_val):,} ({(df_val["label"]=="anomal").mean():.2%} anomal) | '
          f'Test: {len(df_test):,} ({(df_test["label"]=="anomal").mean():.2%} anomal)')

    # ── 5. Normalisierung ──
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(df_train[feature_cols].values)
    val_scaled   = scaler.transform(df_val[feature_cols].values)
    test_scaled  = scaler.transform(df_test[feature_cols].values)

    y_val  = df_val['label'].values
    y_test = df_test['label'].values

    # ── 6. Modell trainieren & Scores berechnen ──
    if cfg.model == 'isolation_forest':
        print(f'  Training Isolation Forest (n={cfg.if_n_estimators}, '
              f'contamination={cfg.if_contamination}) ...')
        # IF braucht 2D Input → kein Windowing
        clf = IsolationForest(
            n_estimators=cfg.if_n_estimators,
            contamination=cfg.if_contamination,
            random_state=42, n_jobs=-1
        )
        clf.fit(train_scaled)
        # decision_function: höher = normaler → invertieren für Anomalie-Score
        scores_val  = -clf.decision_function(val_scaled)
        scores_test = -clf.decision_function(test_scaled)

    else:
        # ── AE-Modelle: Memory-sicheres Station-Wise Training ──
        # Problem: 100 Stationen × 18k Stunden × 24 Window × 19 Features
        # passt nicht als ein numpy-Array in VRAM.
        # Lösung: Bei station-level jede Station einzeln durch den AE,
        # Scores akkumulieren. Training auf einer zufälligen Stichprobe
        # von MAX_TRAIN_SEQS Sequenzen (repräsentativ, VRAM-sicher).
        MAX_TRAIN_SEQS = 50_000   # ~1.5 GB VRAM bei 19 Features, Window=24

        is_lstm = (cfg.model == 'lstm_ae')
        W = cfg.window_size

        def make_sequences_single(scaled, window, is_lstm):
            """Sequenzen für EINE Zeitreihe (Station oder City-Wide)."""
            n = len(scaled)
            if n < window:
                return np.empty((0, window, scaled.shape[1]) if is_lstm
                                else (0, window * scaled.shape[1]))
            idx = np.arange(window, n + 1)
            seqs = np.stack([scaled[i-window:i] for i in idx])  # (n-w, w, f)
            if not is_lstm:
                seqs = seqs.reshape(len(seqs), -1)
            return seqs

        if cfg.aggregation == 'city_wide':
            # City-Wide: eine Zeitreihe → direkt
            X_train_seq = make_sequences_single(train_scaled, W, is_lstm)
            X_val_seq   = make_sequences_single(val_scaled,   W, is_lstm)
            X_test_seq  = make_sequences_single(test_scaled,  W, is_lstm)
            y_val  = y_val[W-1:]
            y_test = y_test[W-1:]
            print(f'  City-Wide Shapes: Train {X_train_seq.shape} | '
                  f'Val {X_val_seq.shape} | Test {X_test_seq.shape}')

            if is_lstm:
                model = LSTMAutoencoder(n_features=n_features, latent_dim=LATENT_DIM)
            else:
                model = VanillaAE(input_dim=W * n_features, latent_dim=LATENT_DIM)
            print(f'  Parameter: {sum(p.numel() for p in model.parameters()):,}')
            model = train_ae(model, X_train_seq, X_val_seq, model_name=cfg.model.upper())

            scores_val  = compute_ae_scores(model, X_val_seq,  scoring=cfg.scoring,
                                            demand_indices=demand_indices,
                                            window_size=W, n_features=n_features)
            scores_test = compute_ae_scores(model, X_test_seq, scoring=cfg.scoring,
                                            demand_indices=demand_indices,
                                            window_size=W, n_features=n_features)

        else:
            # Station-Level: Stationen einzeln verarbeiten
            stations = df_train['station_id'].unique()
            print(f'  Station-Level: {len(stations)} Stationen, '
                  f'max {MAX_TRAIN_SEQS:,} Train-Seqs gesamt')

            # Training: Sequenzen aus allen Stationen sammeln, dann subsampling
            all_train_seqs = []
            for sid in stations:
                mask = df_train['station_id'] == sid
                s_scaled = scaler.transform(df_train[mask][feature_cols].values)
                seqs = make_sequences_single(s_scaled, W, is_lstm)
                if len(seqs) > 0:
                    all_train_seqs.append(seqs)

            X_train_seq = np.concatenate(all_train_seqs, axis=0)
            # Subsampling wenn nötig
            if len(X_train_seq) > MAX_TRAIN_SEQS:
                idx = np.random.choice(len(X_train_seq), MAX_TRAIN_SEQS, replace=False)
                X_train_seq = X_train_seq[idx]
            print(f'  Train-Sequenzen: {len(X_train_seq):,}')

            # Val: ebenfalls station-wise, Labels mitführen
            MAX_VAL_SEQS = 15_000   # Val bleibt klein → kein OOM in train_ae
            val_seqs_list, val_labels_list = [], []
            for sid in df_val['station_id'].unique():
                mask = df_val['station_id'] == sid
                s_scaled = scaler.transform(df_val[mask][feature_cols].values)
                s_labels = df_val[mask]['label'].values
                seqs = make_sequences_single(s_scaled, W, is_lstm)
                if len(seqs) > 0:
                    val_seqs_list.append(seqs)
                    val_labels_list.append(s_labels[W-1:])
            X_val_seq = np.concatenate(val_seqs_list,   axis=0)
            y_val     = np.concatenate(val_labels_list, axis=0)
            # Subsampling Val — stratifiziert (Anomalien behalten)
            if len(X_val_seq) > MAX_VAL_SEQS:
                anom_idx  = np.where(y_val == 'anomal')[0]
                normal_idx = np.where(y_val != 'anomal')[0]
                n_normal  = min(MAX_VAL_SEQS - len(anom_idx), len(normal_idx))
                chosen    = np.concatenate([
                    anom_idx,
                    np.random.choice(normal_idx, n_normal, replace=False)
                ])
                chosen = chosen[np.argsort(chosen)]  # zeitliche Reihenfolge
                X_val_seq = X_val_seq[chosen]
                y_val     = y_val[chosen]
            print(f'  Val-Sequenzen: {len(X_val_seq):,} (davon anomal: {(y_val=="anomal").sum()})')

            # Modell trainieren
            if is_lstm:
                model = LSTMAutoencoder(n_features=n_features, latent_dim=LATENT_DIM)
            else:
                model = VanillaAE(input_dim=W * n_features, latent_dim=LATENT_DIM)
            print(f'  Parameter: {sum(p.numel() for p in model.parameters()):,}')
            model = train_ae(model, X_train_seq, X_val_seq, model_name=cfg.model.upper())
            del X_train_seq  # X_val_seq wird noch für Scoring gebraucht
            torch.cuda.empty_cache()

            # Test: station-wise Scores
            test_seqs_list, test_labels_list = [], []
            for sid in df_test['station_id'].unique():
                mask = df_test['station_id'] == sid
                s_scaled = scaler.transform(df_test[mask][feature_cols].values)
                s_labels = df_test[mask]['label'].values
                seqs = make_sequences_single(s_scaled, W, is_lstm)
                if len(seqs) > 0:
                    test_seqs_list.append(seqs)
                    test_labels_list.append(s_labels[W-1:])
            X_test_seq = np.concatenate(test_seqs_list,   axis=0)
            y_test     = np.concatenate(test_labels_list, axis=0)

            # Val-Scores: direkt auf dem bereits subsamplten X_val_seq
            # y_val ist ebenfalls subgesampelt → Längen passen zusammen
            scores_val  = compute_ae_scores(model, X_val_seq,  scoring=cfg.scoring,
                                            demand_indices=demand_indices,
                                            window_size=W, n_features=n_features)
            scores_test = compute_ae_scores(model, X_test_seq, scoring=cfg.scoring,
                                            demand_indices=demand_indices,
                                            window_size=W, n_features=n_features)
            del X_test_seq, X_val_seq

        model.cpu()
        torch.cuda.empty_cache()

    # ── 7. Evaluation ──
    threshold, _ = find_best_threshold(scores_val, y_val)

    binary_test = (y_test == 'anomal').astype(int)
    preds_test  = (scores_test > threshold).astype(int)

    prec, rec, _ = precision_recall_curve(binary_test, scores_test)
    auc_pr  = auc(rec, prec)
    f1      = f1_score(binary_test, preds_test, zero_division=0)
    try:
        auc_roc = roc_auc_score(binary_test, scores_test)
    except ValueError:
        auc_roc = 0.0

    results = {
        'run_id':         cfg.run_id,
        'description':    cfg.description,
        'experiment_set': cfg.experiment_set,
        'aggregation':    cfg.aggregation,
        'labeling':       cfg.labeling,
        'model':          cfg.model,
        'scoring':        cfg.scoring,
        'feature_set':    cfg.feature_set,
        'auc_pr':         round(auc_pr,  4),
        'f1':             round(f1,      4),
        'auc_roc':        round(auc_roc, 4),
        'threshold':      round(float(threshold), 6),
        'anomaly_rate':   round(float(anomaly_rate), 5),
        'n_anomalies_test': int(binary_test.sum()),
        'n_test':         int(len(binary_test)),
        '_scores_val':    scores_val,   # intern, nicht in CSV
        '_scores_test':   scores_test,
        '_y_test':        y_test,
    }

    print(f'\n  ▶ AUC-PR:  {auc_pr:.4f}  |  F1: {f1:.4f}  |  AUC-ROC: {auc_roc:.4f}')

    # ── 8. In results_log.csv speichern ──
    log_row = {k: v for k, v in results.items() if not k.startswith('_')}
    log_df  = pd.DataFrame([log_row])
    if os.path.exists(RESULTS_LOG):
        existing = pd.read_csv(RESULTS_LOG)
        # Duplikate (run_id) überschreiben
        existing = existing[existing['run_id'] != cfg.run_id]
        log_df = pd.concat([existing, log_df], ignore_index=True)
    log_df.to_csv(RESULTS_LOG, index=False)
    print(f'  ✅ Gespeichert in {RESULTS_LOG}')

    return results


print('run_experiment() definiert. Bereit für Experimente.')

run_experiment() definiert. Bereit für Experimente.


In [31]:
# ══════════════════════════════════════════════════════════════
# 9 — Rohdaten einmalig laden (Basis für alle Runs)
# ══════════════════════════════════════════════════════════════
print('Lade gecleante Demand-Daten ...')
hourly_raw = load_demand_base()
print(f'Geladen: {len(hourly_raw):,} Zeilen | '
      f'{hourly_raw["station_id"].nunique()} Stationen | '
      f'{hourly_raw["hour_ts"].min().date()} – {hourly_raw["hour_ts"].max().date()}')

Lade gecleante Demand-Daten ...
Geladen: 3,067,251 Zeilen | 123 Stationen | 2023-03-31 – 2026-02-02


In [32]:
# ══════════════════════════════════════════════════════════════
# 10 — SET 1: Aggregation (Run 1a Baseline & Run 1b City-Wide)
# ══════════════════════════════════════════════════════════════
results_all = {}

for cfg in [e for e in EXPERIMENTS if e.experiment_set == 'Set1_Aggregation']:
    results_all[cfg.run_id] = run_experiment(cfg, hourly_raw)

# Gewinner Set 1 bestimmen (nach AUC-PR)
set1_results = {k: v for k, v in results_all.items() if 'error' not in v}
winner_set1_id  = max(set1_results, key=lambda k: set1_results[k]['auc_pr'])
winner_set1_agg = set1_results[winner_set1_id]['aggregation']
print(f'\n🏆 SET 1 GEWINNER: Run [{winner_set1_id}] | Aggregation: {winner_set1_agg}')
print(f'   AUC-PR: {set1_results[winner_set1_id]["auc_pr"]:.4f}')


RUN [1a] — Baseline: Station-Level, z-Score, LSTM-AE, MSE
Features: 19 | Aggregation: station
Stationen/Serien: 88 | Zeilen: 2,179,672
Split: Train bis 2025-02-27 | Val bis 2025-08-11 | Test bis 2026-02-02
Labels: {'normal': 2064312, 'grauzone': 108122, 'anomal': 7238} | Anomalie-Rate: 0.0033
Train (normal): 1,392,345 | Val: 323,654 (0.51% anomal) | Test: 351,167 (0.35% anomal)
  Station-Level: 88 Stationen, max 50,000 Train-Seqs gesamt
  Train-Sequenzen: 50,000
  Val-Sequenzen: 15,000 (davon anomal: 1633)
  Parameter: 32,755
  [LSTM_AE] Epoch  10 | Train: 0.37910 | Val: 0.53372
  [LSTM_AE] Epoch  20 | Train: 0.27269 | Val: 0.40876
  [LSTM_AE] Epoch  30 | Train: 0.24705 | Val: 0.37528
  [LSTM_AE] Epoch  40 | Train: 0.23053 | Val: 0.35274
  [LSTM_AE] Epoch  50 | Train: 0.21990 | Val: 0.34033
  [LSTM_AE] Best Val Loss: 0.340326

  ▶ AUC-PR:  0.0290  |  F1: 0.0391  |  AUC-ROC: 0.9145
  ✅ Gespeichert in /content/data/results_log.csv

RUN [1b] — City-Wide Aggregation, z-Score, LSTM-AE, MSE

In [ ]:
# ══════════════════════════════════════════════════════════════
# 11 — SET 2: Labeling (Gewinner-Aggregation aus Set 1)
# ══════════════════════════════════════════════════════════════

# Gewinner-Aggregation in Set-2-Configs einsetzen
for cfg in [e for e in EXPERIMENTS if e.experiment_set == 'Set2_Labeling']:
    cfg.aggregation = winner_set1_agg

for cfg in [e for e in EXPERIMENTS if e.experiment_set == 'Set2_Labeling']:
    results_all[cfg.run_id] = run_experiment(cfg, hourly_raw)

set2_results = {k: v for k, v in results_all.items()
                if 'error' not in v and
                next((e for e in EXPERIMENTS if e.run_id == k and
                      e.experiment_set == 'Set2_Labeling'), None) is not None}
winner_set2_id      = max(set2_results, key=lambda k: set2_results[k]['auc_pr'])
winner_set2_labeling = set2_results[winner_set2_id]['labeling']
print(f'\n🏆 SET 2 GEWINNER: Run [{winner_set2_id}] | Labeling: {winner_set2_labeling}')
print(f'   AUC-PR: {set2_results[winner_set2_id]["auc_pr"]:.4f}')


RUN [2a] — Labeling-Vergleich: z-Score (Referenz)
Features: 19 | Aggregation: station
Stationen/Serien: 88 | Zeilen: 2,179,672
Split: Train bis 2025-02-27 | Val bis 2025-08-11 | Test bis 2026-02-02
Labels: {'normal': 2064312, 'grauzone': 108122, 'anomal': 7238} | Anomalie-Rate: 0.0033
Train (normal): 1,392,345 | Val: 323,654 (0.51% anomal) | Test: 351,167 (0.35% anomal)
  Station-Level: 88 Stationen, max 50,000 Train-Seqs gesamt
  Train-Sequenzen: 50,000
  Val-Sequenzen: 15,000 (davon anomal: 1633)
  Parameter: 32,755
  [LSTM_AE] Epoch  10 | Train: 0.38127 | Val: 0.53978
  [LSTM_AE] Epoch  20 | Train: 0.27218 | Val: 0.41749
  [LSTM_AE] Epoch  30 | Train: 0.24669 | Val: 0.38312
  [LSTM_AE] Epoch  40 | Train: 0.23042 | Val: 0.36330
  [LSTM_AE] Epoch  50 | Train: 0.21958 | Val: 0.34788
  [LSTM_AE] Best Val Loss: 0.347880

  ▶ AUC-PR:  0.0290  |  F1: 0.0391  |  AUC-ROC: 0.9145
  ✅ Gespeichert in /content/data/results_log.csv

RUN [2b] — Labeling: Poisson-ppf(0.9987)
Features: 19 | Aggrega

In [ ]:
# ══════════════════════════════════════════════════════════════
# 12 — SET 3: Modell (Gewinner aus Set 1+2)
# ══════════════════════════════════════════════════════════════

for cfg in [e for e in EXPERIMENTS if e.experiment_set == 'Set3_Model']:
    cfg.aggregation = winner_set1_agg
    cfg.labeling    = winner_set2_labeling

for cfg in [e for e in EXPERIMENTS if e.experiment_set == 'Set3_Model']:
    results_all[cfg.run_id] = run_experiment(cfg, hourly_raw)

set3_results = {k: v for k, v in results_all.items()
                if 'error' not in v and
                next((e for e in EXPERIMENTS if e.run_id == k and
                      e.experiment_set == 'Set3_Model'), None) is not None}
winner_set3_id    = max(set3_results, key=lambda k: set3_results[k]['auc_pr'])
winner_set3_model = set3_results[winner_set3_id]['model']
print(f'\n🏆 SET 3 GEWINNER: Run [{winner_set3_id}] | Modell: {winner_set3_model}')
print(f'   AUC-PR: {set3_results[winner_set3_id]["auc_pr"]:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 13 — SET 4: Scoring (nur für AE-Modelle sinnvoll)
# ══════════════════════════════════════════════════════════════

for cfg in [e for e in EXPERIMENTS if e.experiment_set == 'Set4_Scoring']:
    cfg.aggregation = winner_set1_agg
    cfg.labeling    = winner_set2_labeling
    cfg.model       = winner_set3_model

if winner_set3_model == 'isolation_forest':
    print('⚠️  Gewinner-Modell ist IsolationForest → Scoring-Set nicht anwendbar (kein Reconstruction Error).')
    print('   Set 4 wird übersprungen. Gewinner bleibt MSE (Standard).')
    winner_set4_scoring = 'mse'
else:
    for cfg in [e for e in EXPERIMENTS if e.experiment_set == 'Set4_Scoring']:
        results_all[cfg.run_id] = run_experiment(cfg, hourly_raw)

    set4_results = {k: v for k, v in results_all.items()
                    if 'error' not in v and
                    next((e for e in EXPERIMENTS if e.run_id == k and
                          e.experiment_set == 'Set4_Scoring'), None) is not None}
    winner_set4_id      = max(set4_results, key=lambda k: set4_results[k]['auc_pr'])
    winner_set4_scoring = set4_results[winner_set4_id]['scoring']
    print(f'\n🏆 SET 4 GEWINNER: Run [{winner_set4_id}] | Scoring: {winner_set4_scoring}')
    print(f'   AUC-PR: {set4_results[winner_set4_id]["auc_pr"]:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 14 — Ergebnistabelle & Visualisierung
# ══════════════════════════════════════════════════════════════

results_df = pd.read_csv(RESULTS_LOG)

print('\n' + '='*75)
print('GESAMTERGEBNISSE ALLER RUNS')
print('='*75)
cols_show = ['run_id','experiment_set','aggregation','labeling',
             'model','scoring','auc_pr','f1','auc_roc','anomaly_rate']
print(results_df[cols_show].sort_values('auc_pr', ascending=False).to_string(index=False))

# ── Subplots: Ergebnisse pro Set ──
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Experiment-Framework Ergebnisse — Mannheim BSS', fontsize=13, fontweight='bold')

sets_info = [
    ('Set1_Aggregation', 'aggregation',   'Set 1: Aggregation'),
    ('Set2_Labeling',    'labeling',      'Set 2: Labeling-Methode'),
    ('Set3_Model',       'model',         'Set 3: Modell'),
    ('Set4_Scoring',     'scoring',       'Set 4: Scoring-Funktion'),
]

for ax, (set_name, label_col, title) in zip(axes.flatten(), sets_info):
    subset = results_df[results_df['experiment_set'] == set_name].copy()
    if len(subset) == 0:
        ax.set_visible(False)
        continue

    subset = subset.sort_values('auc_pr', ascending=False)
    x      = range(len(subset))
    labels = [f"{row['run_id']}\n{row[label_col]}" for _, row in subset.iterrows()]

    bars = ax.bar(x, subset['auc_pr'], color=['steelblue' if i == 0 else 'lightsteelblue'
                                               for i in range(len(subset))],
                  edgecolor='white', linewidth=0.5)
    ax2 = ax.twinx()
    ax2.plot(x, subset['auc_roc'], 'rs--', markersize=6, lw=1.5, label='AUC-ROC')
    ax2.set_ylabel('AUC-ROC', color='red', fontsize=9)
    ax2.tick_params(axis='y', labelcolor='red')
    ax2.set_ylim(0, 1.05)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('AUC-PR (Hauptmetrik)')
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')

    # Werte beschriften
    for bar, auc_pr_val in zip(bars, subset['auc_pr']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{auc_pr_val:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/experiment_results_overview.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✅ Plot gespeichert: {PLOTS_DIR}/experiment_results_overview.png')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 15 — PR-Kurven aller Runs im Vergleich
# ══════════════════════════════════════════════════════════════
from sklearn.metrics import precision_recall_curve, auc as sk_auc

completed_runs = {k: v for k, v in results_all.items()
                  if 'error' not in v and '_scores_test' in v}

fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(completed_runs)))

for (run_id, res), color in zip(completed_runs.items(), colors):
    binary = (res['_y_test'] == 'anomal').astype(int)
    if binary.sum() == 0:
        continue
    prec, rec, _ = precision_recall_curve(binary, res['_scores_test'])
    auc_pr = sk_auc(rec, prec)
    cfg_obj = next(e for e in EXPERIMENTS if e.run_id == run_id)
    label = f'[{run_id}] {cfg_obj.aggregation}/{cfg_obj.labeling}/{cfg_obj.model} (AUC={auc_pr:.4f})'
    ax.plot(rec, prec, lw=1.8, color=color, label=label)

# Chance-Level (Anomalie-Rate)
baseline_rate = results_df['anomaly_rate'].mean()
ax.axhline(baseline_rate, color='gray', ls=':', lw=1.2,
           label=f'Random Baseline (Ø Anomalie-Rate: {baseline_rate:.4f})')

ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('PR-Kurven aller Experiment-Runs', fontsize=12, fontweight='bold')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.legend(fontsize=7, loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/pr_curves_all_runs.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ PR-Kurven gespeichert.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 16 — Finale Konfiguration für Cross-City Transfer
# ══════════════════════════════════════════════════════════════

best_run_id = results_df.loc[results_df['auc_pr'].idxmax(), 'run_id']
best_run    = results_df[results_df['run_id'] == best_run_id].iloc[0]

print('='*65)
print('BESTE KONFIGURATION — Basis für Cross-City Transfer (Phase 4)')
print('='*65)
print(f'  Run ID:       {best_run["run_id"]}')
print(f'  Aggregation:  {best_run["aggregation"]}')
print(f'  Labeling:     {best_run["labeling"]}')
print(f'  Modell:       {best_run["model"]}')
print(f'  Scoring:      {best_run["scoring"]}')
print(f'  AUC-PR:       {best_run["auc_pr"]:.4f}')
print(f'  AUC-ROC:      {best_run["auc_roc"]:.4f}')
print(f'  F1:           {best_run["f1"]:.4f}')
print(f'  Anomalie-Rate:{best_run["anomaly_rate"]:.4f}')

# Als JSON für Phase 4 speichern
best_config_path = '/content/data/best_config_for_transfer.json'
best_config_dict = {
    'run_id':       best_run['run_id'],
    'aggregation':  best_run['aggregation'],
    'labeling':     best_run['labeling'],
    'model':        best_run['model'],
    'scoring':      best_run['scoring'],
    'auc_pr':       float(best_run['auc_pr']),
    'auc_roc':      float(best_run['auc_roc']),
    'f1':           float(best_run['f1']),
    'source_city':  CITY,
}
with open(best_config_path, 'w') as f:
    json.dump(best_config_dict, f, indent=2)

print(f'\n✅ Gespeichert: {best_config_path}')
print(f'✅ Alle Ergebnisse: {RESULTS_LOG}')
print(f'\n→ Nächster Schritt: Phase 4 — Cross-City Transfer mit dieser Config')